# LAB2 - Regresion con Spark MLlib (Dataset real)

Nombre y apellidos: Juan Manuel Vega
Grupo: Big Data
Fecha de entrega: 23/03/2026

Notebook de entrega con carga desde CSV, preparacion de features, entrenamiento, prediccion y evaluacion.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder \
    .appName("LAB2_Housing_Regression") \
    .master("local[*]") \
    .getOrCreate()

spark

## 1. Carga del dataset

In [3]:
dataset_path = "housing_spark.csv"
dataset = spark.read.csv(dataset_path, header=True, inferSchema=True)

dataset.show(10, truncate=False)
dataset.printSchema()

+----+--------+---+------+
|size|bedrooms|age|price |
+----+--------+---+------+
|87  |1       |94 |160412|
|155 |2       |17 |346558|
|74  |5       |54 |242441|
|45  |1       |27 |101823|
|288 |5       |3  |708190|
|131 |6       |83 |386656|
|244 |2       |57 |524509|
|172 |1       |97 |325431|
|246 |3       |35 |548094|
|140 |3       |13 |342839|
+----+--------+---+------+
only showing top 10 rows

root
 |-- size: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- price: integer (nullable = true)



## 2. Exploracion del dataset

In [4]:
print("Numero total de registros:", dataset.count())
print("Columnas y tipos:")
for col_name, col_type in dataset.dtypes:
    print(f"- {col_name}: {col_type}")

Numero total de registros: 10000
Columnas y tipos:
- size: int
- bedrooms: int
- age: int
- price: int


## 3. Preparacion de features

In [5]:
assembler = VectorAssembler(
    inputCols=["size", "bedrooms", "age"],
    outputCol="features"
)
data = assembler.transform(dataset).select("features", "price")
data.show(10, truncate=False)

+----------------+------+
|features        |price |
+----------------+------+
|[87.0,1.0,94.0] |160412|
|[155.0,2.0,17.0]|346558|
|[74.0,5.0,54.0] |242441|
|[45.0,1.0,27.0] |101823|
|[288.0,5.0,3.0] |708190|
|[131.0,6.0,83.0]|386656|
|[244.0,2.0,57.0]|524509|
|[172.0,1.0,97.0]|325431|
|[246.0,3.0,35.0]|548094|
|[140.0,3.0,13.0]|342839|
+----------------+------+
only showing top 10 rows



## 4. Division del dataset

In [6]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
print("Registros entrenamiento:", train_data.count())
print("Registros prueba:", test_data.count())

Registros entrenamiento: 8079


Registros prueba: 1921


## 5. Entrenamiento del modelo

In [7]:
lr = LinearRegression(featuresCol="features", labelCol="price")
model = lr.fit(train_data)

print("Coeficientes:", model.coefficients)
print("Intercepto:", model.intercept)

Coeficientes: [1999.1597331638059,25028.002801383947,-397.2577497001458]
Intercepto: -56.0400437369779


## 6. Predicciones

In [8]:
predictions = model.transform(test_data)
predictions.select("features", "price", "prediction").show(20, truncate=False)

+---------------+------+------------------+
|features       |price |prediction        |
+---------------+------+------------------+
|[30.0,1.0,77.0]|54181 |54357.90802564993 |
|[30.0,2.0,50.0]|84611 |90111.8700689378  |
|[30.0,2.0,67.0]|84175 |83358.48832403532 |
|[30.0,3.0,43.0]|109524|117920.67711822275|
|[30.0,3.0,87.0]|93819 |100441.33613141635|
|[30.0,5.0,22.0]|181398|176319.09546469373|
|[30.0,6.0,44.0]|202229|192607.42777267448|
|[31.0,1.0,24.0]|75749 |77411.72849292145 |
|[31.0,3.0,45.0]|118567|119125.32135198629|
|[31.0,3.0,45.0]|124812|119125.32135198629|
|[31.0,4.0,8.0] |164878|158851.8608922756 |
|[31.0,4.0,17.0]|160534|155276.54114497433|
|[31.0,4.0,69.0]|129106|134619.13816056674|
|[31.0,4.0,82.0]|120899|129454.78741446484|
|[31.0,5.0,56.0]|159651|164811.49170805255|
|[31.0,6.0,57.0]|191574|189442.23675973638|
|[32.0,1.0,68.0]|58334 |61931.54723927884 |
|[32.0,5.0,63.0]|159911|164029.84719331536|
|[33.0,1.0,34.0]|73611 |77437.4704622476  |
|[33.0,2.0,31.0]|105315|103657.2

## 7. Evaluacion

In [9]:
evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = evaluator.evaluate(predictions)
print(f"RMSE del modelo: {rmse:.4f}")

RMSE del modelo: 5854.3862


## Preguntas de reflexion

1. **Que variables se utilizan como features?**
Se utilizan `size`, `bedrooms` y `age`.

2. **Cual es la variable objetivo?**
La variable objetivo es `price`.

3. **Por que es necesario crear la columna `features`?**
Porque los algoritmos de Spark MLlib esperan que las variables de entrada esten en una unica columna vectorial llamada normalmente `features`.

4. **Que diferencia hay entre `fit()` y `transform()`?**
`fit()` entrena el modelo con los datos de entrenamiento y `transform()` aplica el modelo entrenado para generar predicciones sobre nuevos datos.

5. **Que indica la metrica RMSE?**
Indica el error promedio de las predicciones en las mismas unidades que la variable objetivo; cuanto menor es el RMSE, mejor ajusta el modelo.